In [1]:
import sys
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from deepagents.middleware.filesystem import FilesystemMiddleware

ModuleNotFoundError: No module named 'deepagents'

In [2]:
from typing import TypedDict

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent


# ---------------------------------------------------------
# Llama 3.2
# ---------------------------------------------------------

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)


# ---------------------------------------------------------
# Filesystem tools
# ---------------------------------------------------------

@tool
def list_files(path: str = ".") -> str:
    """List files recursively under a directory."""
    from pathlib import Path

    files = []

    for p in Path(path).rglob("*"):
        if p.is_file():
            files.append(str(p))

    return "\n".join(files)


@tool
def read_file(path: str) -> str:
    """Read the complete contents of a source file."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


@tool
def write_file(path: str, content: str) -> str:
    """Create or overwrite a file."""
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)

    return f"File written successfully: {path}"


@tool
def run_command(command: str) -> str:
    """Execute a shell command and return stdout/stderr."""
    import subprocess

    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        timeout=120
    )

    return (
        f"Exit code: {result.returncode}\n"
        f"STDOUT:\n{result.stdout}\n"
        f"STDERR:\n{result.stderr}"
    )


# ---------------------------------------------------------
# Agent
# ---------------------------------------------------------

tools = [
    list_files,
    read_file,
    write_file,
    run_command
]

agent = create_react_agent(
    model=llm,
    tools=tools
)



/var/folders/9v/l_th953n4270qgzv2_7ygdd80000gn/T/ipykernel_87740/3410791078.py:83: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [3]:

# ---------------------------------------------------------
# Run
# ---------------------------------------------------------

prompt = """
You are an automated software testing agent.

Your job is to inspect the code under ./target and create
comprehensive automated tests.

Follow these steps:

1. Discover the source files under ./target.
2. Read the relevant source code.
3. Identify all functions and their input types.
4. Create pytest test files under ./tests.
5. Use parameterized tests rather than manually writing
   thousands of individual test cases.

For integer inputs:

- Test zero
- Positive and negative values
- Boundary values
- Values around powers of 10
- Logarithmic magnitudes such as:
  1, 10, 100, 1000, 10000, ...
- For every 10^k also test:
  10^k - 1
  10^k
  10^k + 1

For string inputs:

- Empty string
- Length 1
- Length 10
- Length 100
- Length 1000
- Length 10000
- Continue logarithmically where appropriate
- Test ASCII
- Unicode
- whitespace
- digits
- special characters
- repeated characters

Do not generate huge static test files.
Generate test-data functions or pytest parameterization.

After creating the tests:

6. Run pytest.
7. Analyze failures.
8. Fix the tests if the failure is caused by the test.
9. Run pytest again.
10. Continue until the test suite is working.

Do not modify the production source code.
"""

result = agent.invoke(
    {
        "messages": [
            ("user", prompt)
        ]
    }
)

print(result["messages"][-1].content)

```json
{
  "name": "run_command",
  "parameters": {
    "command": "pytest --junit-xml=report.xml ./tests"
  }
}
```

This command will run pytest in the current directory, generating a JUnit XML report that can be used to analyze test failures. The `--junit-xml` option specifies the output format for the test results.
